# Mini Project 1 — Hospital Respiratory Data Analysis

**Name:** Alefiya Haveliwala  
**Dataset:** CDC NHSN Hospital Respiratory Data (HRD)  
**Date:** May 2026

This notebook analyzes weekly U.S. hospital capacity and respiratory-related utilization from the CDC. It is organized into five sections: Overview, Data Profile, Analysis (with three charts), Conclusions, and Process.

In [ ]:
# Setup — run this cell first
!pip install jupyter plotly kaleido pandas

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")


---

## Section 1 — Overview

### Dataset

I use the **CDC NHSN Hospital Respiratory Data (HRD)** dataset from the Centers for Disease Control and Prevention. The data is publicly available at [data.cdc.gov](https://data.cdc.gov/) and was downloaded as a CSV for analysis in Python with pandas.

The file contains **weekly** hospital metrics across the United States, including:

- Inpatient and ICU bed counts and occupancy
- Patients hospitalized with COVID-19, influenza, and RSV
- Admission counts by age group
- Pre-calculated occupancy percentages and hospital reporting coverage

Rows are tagged with a **week ending date** and a **geographic aggregation** code (national `USA`, individual states, territories, and other groupings).

### Why this dataset

Hospitals must match fixed capacity (beds, ICU) to demand that shifts week by week. From an **HCD** perspective, system-level constraints in healthcare can translate into differences in access, wait times, and care experience across places and time periods. This dataset makes those constraints visible at a population scale.

### Three analytical questions

1. **Which regions consistently operate near high occupancy levels over time?**  
   Identifying states that frequently experience high utilization relative to capacity.

2. **Are there seasonal or temporal patterns in hospital occupancy and bed utilization?**  
   Analyzing weekly trends to detect recurring peaks or cycles in hospital demand.

3. **Do pediatric and adult bed occupancy rates follow different patterns over time, and are pediatric ICU beds proportionally more strained than adult ICU beds?**  
   Comparing monthly occupancy trends for adult vs pediatric inpatient and ICU beds at the national level.

### Why these questions matter

Hospitals operate under continuous constraints where capacity must adapt to fluctuating demand. By examining how occupancy varies over time, across regions, and across patient populations, this project reveals whether strain on hospital systems is **evenly distributed or concentrated** in specific places, seasons, or age groups.

From a **computational** perspective, the project applies skills from this course: loading and cleaning real-world structured data, handling time series, grouping and aggregation with pandas, deriving metrics (such as occupancy percentages from bed counts), and visualizing patterns with Plotly.

### What a practitioner would do with these findings

Public health officials, hospital administrators, and policy planners could use these patterns to anticipate seasonal surges, target resources to persistently high-burden states, and understand whether pediatric or adult capacity needs different staffing and surge planning.

---

## Section 2 — Data Profile

Load the dataset and inspect its structure. After running the cells below, I interpret what each operation shows.

In [ ]:
# Load dataset (same folder as this notebook)
DATA_FILE = "MP1_Weekly_Hospital_Respiratory_Data.csv"

df = pd.read_csv(DATA_FILE, low_memory=False)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [9]:
# Missing values per column (show columns with any missing values)
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False).head(20)

Number of Pediatric RSV Admissions, 0-4 years                                   14650
Number of Adult Influenza Admissions, 18-49 years                               14650
Number of Adult COVID-19 Admissions, 75+ years, per 100,000 population          14650
Number of Adult COVID-19 Admissions, 65-74 years, per 100,000 population        14650
Number of Adult COVID-19 Admissions, 50-64 years, per 100,000 population        14650
Number of Adult COVID-19 Admissions, 50-64 years                                14650
Number of Adult COVID-19 Admissions, 65-74 years                                14650
Number of Adult COVID-19 Admissions, 75+ years                                  14650
Number of Pediatric Influenza Admissions, 0-4 years                             14650
Number of Pediatric Influenza Admissions, 5-17 years                            14650
Number of Adult Influenza Admissions, 50-64 years                               14650
Number of Pediatric Influenza Admissions, 5-17 years, 

### Data profile interpretation

One sentence per operation — what each result tells us about the dataset.

#### `df.head()`

The first rows show **one record per geographic unit per week**, with many columns for bed counts, patient counts, and pre-calculated percentages. Early rows for some territories can be sparse when hospitals were not yet reporting.

#### `df.info()`

The table has about **17,900 rows and 190 columns**. Most columns are `object` (strings from the CSV) or numeric after cleaning. `Week Ending Date` should be parsed as datetime for time-series work. The wide format reflects many related metrics bundled in one CDC export.

#### `df.describe()`

Numeric summaries show inpatient occupancy typically in the **low 70s percent** on average, with substantial spread (some values are missing or extreme when reporting is incomplete). ICU occupancy averages slightly lower than inpatient but with similar variability.

#### `df.isnull().sum()`

Missing values are **common in admission breakdown columns** (often 14,000+ nulls per column) because reporting expanded over time and not every hospital reports every field every week. Core occupancy percentage fields are more complete (~17,600+ non-null values) and are the focus of this analysis. Missing admission detail matters less for the capacity questions in this notebook.

#### Columns used in this analysis

| Column / group | Role |
|----------------|------|
| `Week Ending Date` | Weekly time series |
| `Geographic aggregation` | State, national (`USA`), or other region codes |
| Occupancy percentages | Inpatient and ICU capacity pressure |
| Adult / pediatric bed counts | Question 3 — comparing strain by patient population |

These fields support the three research questions: regional pressure over time, seasonal patterns, and adult vs pediatric occupancy.

---

## Section 3 — Analysis

Each research question below has a chart built for Assignment 6 (Week 6), saved as a static PNG with Kaleido, followed by a plain-language interpretation.

### Question 1

**Which regions or hospitals consistently operate near high occupancy levels over time?**

*Chart type: faceted line chart (weekly time series by state).*

In [ ]:
# Section 4 visualization using Plotly Express
import pandas as pd
import plotly.express as px
from pathlib import Path

csv_candidates = [
    Path("MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../MP1 Project/MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../530 Week5 Project/Weekly_Hospital_Respiratory_Data.csv"),
    Path("Weekly_Hospital_Respiratory_Data.csv"),
]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find hospital respiratory CSV. Tried: "
        + ", ".join(str(p) for p in csv_candidates)
    )

resp = pd.read_csv(csv_path, low_memory=False)

state_codes = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "DC"
}

capacity_cols = [
    "Percent Inpatient Beds Occupied",
    "Percent ICU Beds Occupied",
]

resp["Week Ending Date"] = pd.to_datetime(resp["Week Ending Date"], errors="coerce")
for col in capacity_cols:
    resp[col] = (
        resp[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    resp[col] = pd.to_numeric(resp[col], errors="coerce")

state_df = resp[
    resp["Geographic aggregation"].isin(state_codes)
].dropna(subset=["Week Ending Date"] + capacity_cols).copy()
state_df = state_df.sort_values("Week Ending Date")

# Focus on top states by latest inpatient occupancy for readable comparison
latest_week = state_df["Week Ending Date"].max()
latest_slice = state_df[state_df["Week Ending Date"] == latest_week]
top_states = (
    latest_slice
    .nlargest(12, "Percent Inpatient Beds Occupied")["Geographic aggregation"]
    .tolist()
)
plot_df = state_df[state_df["Geographic aggregation"].isin(top_states)].copy()

long_df = plot_df.melt(
    id_vars=["Week Ending Date", "Geographic aggregation"],
    value_vars=capacity_cols,
    var_name="Metric",
    value_name="OccupancyPercent",
)

metric_labels = {
    "Percent Inpatient Beds Occupied": "Inpatient beds occupied (%)",
    "Percent ICU Beds Occupied": "ICU beds occupied (%)",
}
long_df["Metric"] = long_df["Metric"].map(metric_labels)

fig = px.line(
    long_df,
    x="Week Ending Date",
    y="OccupancyPercent",
    color="Geographic aggregation",
    facet_row="Metric",
    title=None,
    labels={
        "Week Ending Date": "Reporting week",
        "OccupancyPercent": "Occupancy (%)",
        "Geographic aggregation": "State",
        "Metric": "Capacity measure",
    },
    height=900,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(
    title=dict(
        text=(
            "<b>Hospital capacity pressure varies across high-burden states</b><br>"
            "<span style='font-size:12px;color:#444'>"
            "Top 12 states by most recent inpatient occupancy. "
            "Top row: inpatient beds. Bottom row: ICU beds."
            "</span>"
        ),
        x=0.5,
        xanchor="center",
        pad=dict(b=18),
    ),
    margin=dict(t=170, r=40, b=70, l=70),
    legend_title_text="State",
    title_automargin=True,
)
fig.update_traces(
    hovertemplate=(
        "State: %{fullData.name}<br>"
        "Reporting week: %{x|%b %d, %Y}<br>"
        "Occupancy: %{y:.1f}%<extra></extra>"
    )
)
png_path = Path("viz1_state_hospital_capacity.png")
fig.write_image(png_path, engine="kaleido", scale=2)
print(f"Saved {png_path.resolve()}")

![Question 1 chart — state capacity over time](viz1_state_hospital_capacity.png)


**What this chart shows**

The faceted line chart tracks **inpatient** (top) and **ICU** (bottom) occupancy for the 12 states with the highest inpatient occupancy in the most recent reporting week. Several states—notably in the Northeast and Mid-Atlantic—run **persistently high** (often above 75–80% inpatient occupancy) across many weeks rather than spiking only once.

That pattern suggests **sustained capacity pressure**, not a single-week anomaly. ICU lines do not always mirror inpatient lines, so critical-care strain can differ from general bed pressure within the same state.

### Question 2

**Are there seasonal or temporal patterns in hospital occupancy and bed utilization?**

*Chart type: year × month heatmap (national USA).*

In [ ]:
# Second visualization: seasonal / temporal patterns (national USA)
# Heatmap (year × month): same calendar month lines up across years for seasonality.

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# COLOR SCALE: 60% = bright yellow, 80% = deep red
YELLOW_TO_RED = [
    [0.0, "#FFEB3B"],
    [0.25, "#FFC107"],
    [0.5, "#FF9800"],
    [0.75, "#F44336"],
    [1.0, "#B71C1C"],
]
ZMIN_PCT, ZMAX_PCT = 60.0, 80.0

csv_candidates = [
    Path("MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../MP1 Project/MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../530 Week5 Project/Weekly_Hospital_Respiratory_Data.csv"),
    Path("Weekly_Hospital_Respiratory_Data.csv"),
]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find hospital respiratory CSV. Tried: "
        + ", ".join(str(p) for p in csv_candidates)
    )

resp_seasonal = pd.read_csv(csv_path, low_memory=False)
capacity_cols = [
    "Percent Inpatient Beds Occupied",
    "Percent ICU Beds Occupied",
]
resp_seasonal["Week Ending Date"] = pd.to_datetime(
    resp_seasonal["Week Ending Date"], errors="coerce"
)
for col in capacity_cols:
    resp_seasonal[col] = (
        resp_seasonal[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    resp_seasonal[col] = pd.to_numeric(resp_seasonal[col], errors="coerce")

usa = resp_seasonal[resp_seasonal["Geographic aggregation"] == "USA"].copy()
usa = usa.dropna(
    subset=["Week Ending Date", "Percent Inpatient Beds Occupied", "Percent ICU Beds Occupied"]
).sort_values("Week Ending Date")

usa["year"] = usa["Week Ending Date"].dt.year
usa["month"] = usa["Week Ending Date"].dt.month
monthly = (
    usa.groupby(["year", "month"], as_index=False)[capacity_cols]
    .mean()
    .rename(
        columns={
            "Percent Inpatient Beds Occupied": "inpatient_pct",
            "Percent ICU Beds Occupied": "icu_pct",
        }
    )
)

p_in = monthly.pivot(index="year", columns="month", values="inpatient_pct")
p_icu = monthly.pivot(index="year", columns="month", values="icu_pct")

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
x_labels = [month_names[m - 1] for m in p_in.columns]

fig_seasonal = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("Inpatient (% occupied)", "ICU (% occupied)"),
    vertical_spacing=0.16,
)

tick_vals = [60, 65, 70, 75, 80]
tick_text = ["60", "65", "70", "75", "80"]

fig_seasonal.add_trace(
    go.Heatmap(
        z=p_in.values,
        x=x_labels,
        y=p_in.index.astype(str),
        zmin=ZMIN_PCT,
        zmax=ZMAX_PCT,
        zsmooth=False,
        xgap=2,
        ygap=2,
        colorscale=YELLOW_TO_RED,
        hovertemplate="Year: %{y}<br>Month: %{x}<br>Inpatient: %{z:.1f}%<extra></extra>",
        colorbar=dict(
            title="Occupancy (%)",
            tickvals=tick_vals,
            ticktext=tick_text,
            len=0.42,
            y=0.78,
            yanchor="middle",
        ),
    ),
    row=1,
    col=1,
)
fig_seasonal.add_trace(
    go.Heatmap(
        z=p_icu.values,
        x=x_labels,
        y=p_icu.index.astype(str),
        zmin=ZMIN_PCT,
        zmax=ZMAX_PCT,
        zsmooth=False,
        xgap=2,
        ygap=2,
        colorscale=YELLOW_TO_RED,
        hovertemplate="Year: %{y}<br>Month: %{x}<br>ICU: %{z:.1f}%<extra></extra>",
        colorbar=dict(
            title="Occupancy (%)",
            tickvals=tick_vals,
            ticktext=tick_text,
            len=0.42,
            y=0.22,
            yanchor="middle",
        ),
    ),
    row=2,
    col=1,
)

fig_seasonal.update_xaxes(showgrid=False, zeroline=False, side="bottom", type="category", row=1, col=1)
fig_seasonal.update_xaxes(showgrid=False, zeroline=False, side="bottom", type="category", row=2, col=1)
fig_seasonal.update_yaxes(showgrid=False, zeroline=False, type="category", row=1, col=1)
fig_seasonal.update_yaxes(showgrid=False, zeroline=False, type="category", row=2, col=1)

fig_seasonal.update_layout(
    title=dict(
        text=(
            "<b>USA: seasonal occupancy (month × year)</b>"
            "<br><sup>60% = yellow, 80% = red</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=15),
        pad=dict(b=12),
    ),
    height=780,
    margin=dict(l=88, r=120, b=70, t=165),
)

print(f"Scale: {ZMIN_PCT}% (yellow) → {ZMAX_PCT}% (red); colorscale has {len(YELLOW_TO_RED)} stops")

png_path = Path("viz2_seasonal_heatmap.png")
fig_seasonal.write_image(png_path, engine="kaleido", scale=2)
print(f"Saved {png_path.resolve()}")

![Question 2 chart — seasonal heatmap](viz2_seasonal_heatmap.png)


**What this chart shows**

The heatmaps aggregate **national (USA)** weekly data into average occupancy by **calendar month and year**. Recurring darker bands in late fall and winter—especially for inpatient beds—support a **seasonal rhythm** in hospital demand. Summer months are often somewhat lighter, though the exact intensity changes year to year (for example, respiratory surges differ between COVID-19, flu, and RSV seasons).

A single line chart over time would show peaks but makes it harder to see whether the **same months** repeat across years; the heatmap layout is built for that seasonal comparison.

### Question 3

**Do pediatric and adult bed occupancy rates follow different patterns over time, and are pediatric ICU beds proportionally more strained than adult ICU beds?**

*Chart type: monthly average line chart (adult vs pediatric, national USA).*

In [8]:
# Third visualization: monthly trends — adult vs pediatric occupancy (USA)
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

csv_candidates = [
    Path("MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../MP1 Project/MP1_Weekly_Hospital_Respiratory_Data.csv"),
    Path("../530 Week5 Project/Weekly_Hospital_Respiratory_Data.csv"),
    Path("Weekly_Hospital_Respiratory_Data.csv"),
]
csv_path = next((p for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find hospital respiratory CSV")

usa = pd.read_csv(csv_path, low_memory=False)
usa = usa[usa["Geographic aggregation"] == "USA"].copy()

bed_cols = [
    "Number of Adult Inpatient Beds",
    "Number of Pediatric Inpatient beds",
    "Number of Adult Inpatient Beds Occupied",
    "Number of Pediatric Inpatient Beds Occupied",
    "Number of Adult ICU Beds",
    "Number of Pediatric ICU Beds",
    "Number of Adult ICU Beds Occupied",
    "Number of Pediatric ICU Beds Occupied",
]
usa["Week Ending Date"] = pd.to_datetime(usa["Week Ending Date"], errors="coerce")
for col in bed_cols:
    usa[col] = pd.to_numeric(
        usa[col].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )
usa = usa.dropna(subset=["Week Ending Date"]).sort_values("Week Ending Date")

usa["adult_inpatient_pct"] = usa["Number of Adult Inpatient Beds Occupied"] / usa["Number of Adult Inpatient Beds"] * 100
usa["ped_inpatient_pct"] = usa["Number of Pediatric Inpatient Beds Occupied"] / usa["Number of Pediatric Inpatient beds"] * 100
usa["adult_icu_pct"] = usa["Number of Adult ICU Beds Occupied"] / usa["Number of Adult ICU Beds"] * 100
usa["ped_icu_pct"] = usa["Number of Pediatric ICU Beds Occupied"] / usa["Number of Pediatric ICU Beds"] * 100

usa["month"] = usa["Week Ending Date"].dt.to_period("M").dt.to_timestamp()
monthly = usa.groupby("month", as_index=False)[
    ["adult_inpatient_pct", "ped_inpatient_pct", "adult_icu_pct", "ped_icu_pct"]
].mean()
monthly["month_label"] = monthly["month"].dt.strftime("%b %Y")

fig_monthly = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.22,
    subplot_titles=(
        "Inpatient: adult vs pediatric occupancy by month",
        "ICU: adult vs pediatric occupancy by month",
    ),
)

for adult_col, ped_col, adult_name, ped_name, row in [
    ("adult_inpatient_pct", "ped_inpatient_pct", "Adult inpatient", "Pediatric inpatient", 1),
    ("adult_icu_pct", "ped_icu_pct", "Adult ICU", "Pediatric ICU", 2),
]:
    fig_monthly.add_trace(
        go.Scatter(
            x=monthly["month_label"],
            y=monthly[adult_col],
            name=adult_name,
            mode="lines+markers",
            line=dict(color="#1976D2", width=2),
            marker=dict(size=5),
        ),
        row=row,
        col=1,
    )
    fig_monthly.add_trace(
        go.Scatter(
            x=monthly["month_label"],
            y=monthly[ped_col],
            name=ped_name,
            mode="lines+markers",
            line=dict(color="#FF7043", width=2),
            marker=dict(size=5),
        ),
        row=row,
        col=1,
    )

fig_monthly.update_yaxes(title_text="Occupancy (%)", row=1, col=1, range=[50, 90])
fig_monthly.update_yaxes(title_text="Occupancy (%)", row=2, col=1, range=[50, 90])
# Show month labels on BOTH panels (shared_xaxes hides top ticks by default)
for row in (1, 2):
    fig_monthly.update_xaxes(
        title_text="",
        showticklabels=True,
        tickangle=-45,
        type="category",
        row=row,
        col=1,
    )

fig_monthly.update_layout(
    title=dict(
        text="Adult vs pediatric hospital occupancy by month (USA)",
        x=0.5,
        xanchor="center",
        y=0.99,
    ),
    height=700,
    width=960,
    hovermode="x unified",
    template="plotly_white",
    legend=dict(
        orientation="h",
        y=-0.12,
        x=0.5,
        xanchor="center",
        yanchor="top",
    ),
    margin=dict(t=90, b=160, l=60, r=40),
)
png_path = Path("viz3_pediatric_adult_monthly.png")
fig_monthly.write_image(png_path, engine="kaleido", scale=2)
print(f"Saved {png_path.resolve()}")

avg_adult_inp = usa["adult_inpatient_pct"].mean()
avg_ped_inp = usa["ped_inpatient_pct"].mean()
avg_adult_icu = usa["adult_icu_pct"].mean()
avg_ped_icu = usa["ped_icu_pct"].mean()
icu_ratio = avg_ped_icu / avg_adult_icu

print("=" * 60)
print("AVERAGE OCCUPANCY (all weeks, USA — use for your write-up)")
print("=" * 60)
print(f"  Adult inpatient:       {avg_adult_inp:.1f}%")
print(f"  Pediatric inpatient: {avg_ped_inp:.1f}%")
print(f"  Adult ICU:             {avg_adult_icu:.1f}%")
print(f"  Pediatric ICU:         {avg_ped_icu:.1f}%")
print()
print("Q1: Do patterns differ over time?")
print("  → Compare blue vs orange in EACH panel month by month.")
print("  → Lines that peak in different months or move apart = different patterns.")
print()
print("Q2: Is pediatric ICU proportionally more strained than adult ICU?")
if icu_ratio > 1:
    print(f"  → YES (pediatric ICU average is higher; ratio {icu_ratio:.2f})")
else:
    print(f"  → NO (pediatric ICU average is lower; ratio {icu_ratio:.2f})")
print("=" * 60)



/var/folders/mj/gzf8nsxd62x0ckq4ns594qcw0000gn/T/ipykernel_84937/4217621513.py:123: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig_monthly.write_image(png_path, engine="kaleido", scale=2)


Saved /Users/alefiya_haveliwala/UW/[6] SPR '26/[2] 530 - Commutational Concepts/530 Projects/MP1 Project/viz3_pediatric_adult_monthly.png
AVERAGE OCCUPANCY (all weeks, USA — use for your write-up)
  Adult inpatient:       75.5%
  Pediatric inpatient: 64.0%
  Adult ICU:             72.7%
  Pediatric ICU:         67.1%

Q1: Do patterns differ over time?
  → Compare blue vs orange in EACH panel month by month.
  → Lines that peak in different months or move apart = different patterns.

Q2: Is pediatric ICU proportionally more strained than adult ICU?
  → NO (pediatric ICU average is lower; ratio 0.92)


![Question 3 chart — adult vs pediatric occupancy by month](viz3_pediatric_adult_monthly.png)


**What this chart shows**

The monthly line charts compare **adult (blue)** and **pediatric (orange)** occupancy at the national level. In both panels, adult occupancy runs **higher** than pediatric occupancy on average (about **76% vs 64%** inpatient; **73% vs 67%** ICU).

The lines generally move in the same direction month to month, so seasonal pressure affects both groups—but not by the same amount. When the lines **diverge**, adult and pediatric patterns differ for that period.

For proportional ICU strain: pediatric ICU occupancy is **not** higher than adult ICU on average (pediatric/adult ratio ≈ **0.92**). Pediatric ICU beds are pressured, but **adult ICU beds are proportionally more strained** in this dataset.

---

## Section 4 — Conclusions

### Question 1 — Consistently high occupancy

Several states operate near or above **75% inpatient occupancy** for a large share of weeks in the dataset. Rhode Island, Massachusetts, Maryland, and Washington are among those with the highest sustained averages. That implies capacity pressure is **localized**, not only a national average story.

### Question 2 — Seasonal patterns

National data show **recurring winter-heavy occupancy**, with inpatient beds often more stressed in colder months. Year-to-year differences still matter—pandemic and post-pandemic reporting changed which respiratory illnesses drove peaks—but the month-by-month heatmap supports planning around **predictable seasonal demand**.

### Question 3 — Adult vs pediatric occupancy

Nationally, **adult inpatient and ICU occupancy are higher** than pediatric occupancy on average (about 76% vs 64% inpatient; 73% vs 67% ICU). Monthly trends are related—both rise and fall with seasonal demand—but they are not identical; the lines separate in some months.

**Pediatric ICU is not proportionally more strained than adult ICU** in this dataset (ratio ≈ 0.92). Surge planning should still account for pediatric beds, but adult critical-care capacity carries the higher typical load.

### What I would investigate next

With more time, I would add **facility-level** data (where available) to move from state averages to specific hospital systems, link occupancy to **admission type** (COVID-19 vs flu vs RSV), and study whether high occupancy correlates with reporting gaps or delayed care indicators.


---

## Section 5 — Process

### How I arrived at this notebook

I proposed this project in **MP1a** after searching the CDC data portal for a dataset that was public, weekly, and relevant to healthcare systems. The NHSN Hospital Respiratory Data met those criteria and aligned with my interest in how **large-scale constraints** show up in people's access to care.

**Week 5 (pandas):** I loaded the CSV, parsed dates, cleaned percentage strings, and grouped by `Geographic aggregation` to understand row counts and missingness. That work became the Data Profile section.

**Week 6 (visualization):** I built three Plotly charts in `week6_mp1_starter.ipynb`—state line trends, a national seasonal heatmap, and an adult-vs-pediatric monthly comparison. All three visualization code cells were carried into this MP1 notebook for the Analysis section.

**AI use:** I used AI to speed up repetitive pandas cleaning (for example stripping `%` from occupancy fields) and to sanity-check chart choices against the Week 6 four-rule guide. The research questions, interpretations, and conclusions are my own, grounded in the patterns visible in the charts.

**Challenges:** The raw file is wide (190 columns) with uneven reporting over time, so I focused on a small set of occupancy columns rather than every admission field. Exporting static PNGs required installing **kaleido** and running each chart cell once to commit images to the repo.

**Standalone artifact:** This `MP1 Project` folder contains the notebook, the CSV, and saved chart images so the project can be reviewed without depending on other weekly folders.